# RootBot: детектор фігурок на GPU (v6 + синтетика v7)

**Один раз:** у Colab ліворуч ключик **Secrets** -> `KAGGLE_API_TOKEN` = твій Kaggle-токен, увімкнути доступ для ноутбука.
Дані: приватний Kaggle-датасет `natalyakoz/rootbot-datasets` (zip-и v6 і v7, завантажити командою з `scripts/detector/README.md`).

1. **Runtime -> Change runtime type -> T4 GPU**
2. **Runtime -> Run all**. Ваги й метрики пишуться на Google Drive (`MyDrive/rootbot_detector`) після кожної епохи,
   тож якщо Colab відключиться - просто знову **Run all**: навчання продовжиться з того місця.

Що робить: три кандидати (див. `RUNS`), вибір **лише на val**, потім **один раз** test, порівняння з `v5_all_s0` команди.

In [ ]:
!pip install -q -U ultralytics kaggle
import os, subprocess
from google.colab import userdata, drive
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/rootbot_detector'; os.makedirs(OUT, exist_ok=True)
!rm -rf /content/repo && git clone -q -b vision/dataset-pipeline https://github.com/olenakshyk/RootBot67 /content/repo
!mkdir -p /content/data && cd /content/data && [ -d v6_2026-11-09 ] || (kaggle datasets download natalyakoz/rootbot-datasets -p . -q && while ls *.zip >/dev/null 2>&1; do for z in *.zip; do unzip -q -o "$z" && rm "$z"; done; done; ls)

In [ ]:
# lists: v6 train (human frames x3), val / test only human frames; synthetic list for pre-training
!python /content/repo/scripts/detector/make_lists.py /content/data/v6_2026-11-09 /content/lists
import glob, yaml
syn = sorted(glob.glob('/content/data/v7_simulation_2026-11-10/synthetic_simulation/yolo/images/train/*.jpg'))
open('/content/lists/synthetic.txt', 'w').write('\n'.join(syn) + '\n')
d = yaml.safe_load(open('/content/lists/data.yaml')); d['train'] = 'synthetic.txt'
yaml.safe_dump(d, open('/content/lists/data_synthetic.yaml', 'w'), sort_keys=False)
print(len(syn), 'synthetic frames')

In [ ]:
from pathlib import Path
from ultralytics import YOLO
AUG = dict(degrees=10, flipud=0.5, fliplr=0.5, hsv_h=0.01, cos_lr=True, close_mosaic=10, save_period=5, seed=0, workers=4)

def train(name, weights, data, **kw):
    run = Path(OUT) / 'runs' / name
    if (run / 'weights/last.pt').exists() and not (run / 'done').exists():
        YOLO(run / 'weights/last.pt').train(resume=True)          # Colab disconnected: continue
    elif not (run / 'done').exists():
        YOLO(weights).train(data=data, project=f'{OUT}/runs', name=name, exist_ok=True, **AUG, **kw)
    (run / 'done').touch()
    return run / 'weights/best.pt'

REAL, SYN = '/content/lists/data.yaml', '/content/lists/data_synthetic.yaml'
RUNS = {}
# A: same recipe as the team's v5_all_s0 but bigger model and longer (baseline: YOLO11s, 40 ep, 1280)
RUNS['m1280_real'] = train('m1280_real', 'yolo11m.pt', REAL, imgsz=1280, epochs=80, batch=8, patience=30)
# B: pre-train on 10k synthetic frames, then fine-tune on real ones
syn = train('m960_synthetic', 'yolo11m.pt', SYN, imgsz=960, epochs=6, batch=16, patience=100, close_mosaic=0)
RUNS['m1280_syn_real'] = train('m1280_syn_real', str(syn), REAL, imgsz=1280, epochs=60, batch=8, patience=30)
# C: larger input for the tiny pieces on scanner frames (weak spot of v5_all_s0: recall 0.79)
RUNS['m1600_real'] = train('m1600_real', 'yolo11m.pt', REAL, imgsz=1600, epochs=60, batch=4, patience=30)

In [ ]:
# choose ONLY on val (19 frames); test stays untouched until the end
import json
EV = '/content/repo/scripts/detector/evaluate.py'
val = {}
for name, w in RUNS.items():
    imgsz = 1600 if '1600' in name else 1280
    for tiles in ('', '--tiles'):
        key = f'{name}{tiles}'
        subprocess.run(f'python {EV} predict {w} /content/lists/val.txt {OUT}/val_{key}.json --imgsz {imgsz} {tiles}', shell=True, check=True)
        r = json.loads(subprocess.run(f'python {EV} score {OUT}/val_{key}.json --boot 500', shell=True, capture_output=True, text=True).stdout)
        m = YOLO(w).val(data=REAL, split='val', imgsz=imgsz, verbose=False, plots=False)
        val[key] = dict(P=r['all']['P'], R=r['all']['R'], count_exact=r['all']['count_exact'], mAP50=float(m.box.map50), mAP50_95=float(m.box.map))
        print(key, {k: round(v, 3) for k, v in val[key].items()})
# selection rule decided in advance: mean of team P and R (the robot needs both), ties -> mAP50-95
best = max(val, key=lambda k: ((val[k]['P'] + val[k]['R']) / 2, val[k]['mAP50_95']))
print('chosen on val:', best)
json.dump(val, open(f'{OUT}/val_results.json', 'w'), indent=1)

In [ ]:
# test: once, for the chosen model (all candidates are also scored, but only for the record)
name, tiles = best.replace('--tiles', ''), '--tiles' if best.endswith('--tiles') else ''
w, imgsz = RUNS[name], 1600 if '1600' in name else 1280
subprocess.run(f'python {EV} predict {w} /content/lists/test.txt {OUT}/test_{best}.json --imgsz {imgsz} {tiles}', shell=True, check=True)
!python {EV} score {OUT}/test_{best}.json --out {OUT}/test_{best}_score.json
m = YOLO(w).val(data=REAL, split='test', imgsz=imgsz, project=OUT, name='test_val', exist_ok=True)
print('test mAP50', round(float(m.box.map50), 3), 'mAP50-95', round(float(m.box.map), 3))
print('team baseline v5_all_s0 on the same test: P 0.969 R 0.920, mAP50 0.948, mAP50-95 0.580')
!cp {w} {OUT}/rootbot_detector_best.pt